### 🧭 **Topic Modeling**

Este guia descreve um fluxo prático e reproduzível para modelagem de tópicos em coleções de texto.  
A ideia é evitar treinar modelos pesados desde o início, ajustando primeiro os hiperparâmetros em uma amostra menor e depois escalando para o conjunto completo.

---

#### 📑 **Fluxo Geral**

1. **Separar uma amostra** para ajuste de parâmetros.  
2. **Treinar o modelo** em uma parte maior do conjunto usando os parâmetros ajustados.  
3. **Classificar o restante do corpus** com os tópicos aprendidos.

---


In [2]:
'''Carrega o CSV, removendo linhas mal formatadas'''
import pandas as pd

file_path = "../data/preprocessed_english_titles.csv"
data_frame = pd.read_csv(file_path,engine = 'python', on_bad_lines='skip')
print(f"Total documents loaded: {len(data_frame)}")

Total documents loaded: 686626


In [3]:
'''Remoção de Stopwords'''
from PreProcessing.pre_processing import PreProcessing
from nltk.corpus import stopwords

pp = PreProcessing(language="en")

custom_stopwords = [line.strip() for line in open('stopwords.txt', 'r', encoding='utf-8')]
english_stopwords = set(stopwords.words('english'))
pp.append_stopwords_list(list(english_stopwords - set(pp.stopwords)) + custom_stopwords)

data_frame["clean_text"] = data_frame["clean_text"].apply(pp.remove_stopwords)


In [4]:
'''Remoção de linhas nulas e duplicatas'''
nan_count = data_frame['clean_text'].isna().sum()
df = data_frame[data_frame['clean_text'].notna()].reset_index(drop=True)
print(f"{nan_count} rows with NaN in 'clean_text' were removed.")

duplicate_count = df.duplicated(subset=['clean_text']).sum()
df.drop_duplicates(subset=['clean_text'], inplace=True)
print(f"{duplicate_count} duplicate rows based on 'clean_text' were removed.")

print(f"Total documents after cleaning: {len(df)}")

0 rows with NaN in 'clean_text' were removed.
12991 duplicate rows based on 'clean_text' were removed.
Total documents after cleaning: 673635


In [5]:
'''Amostra com 1% dos dados'''
sample_df = df.sample(frac=0.01, random_state=42)
sample_docs = sample_df['clean_text'].tolist()
print(f"Sampled {len(sample_docs)} documents for topic modeling.")

Sampled 6736 documents for topic modeling.


In [11]:
# Defining parameters for topic modeling

"""
UMAP PARAMETERS

n_neighbors: Controla o equilíbrio entre a preservação da estrutura global e local dos dados.
n_components: A dimensionalidade do espaço onde os clusters são formados. 
              Um espaço de menor dimensão pode forçar os pontos a se agruparem de forma mais densa.

HDBSCAN PARAMETERS
min_cluster_size: O tamanho mínimo de um cluster. Clusters menores que esse valor serão considerados ruído.
min_samples: Influencia a sensibilidade do algoritmo à densidade dos pontos.
             Valores maiores consideram apenas áreas muito densas como clusters.
cluster_eps: Define a distância máxima entre pontos para que sejam considerados parte do mesmo cluster.
             
EMBEDDING MODELS
Modelos de linguagem pré-treinados usados para gerar embeddings dos textos.
"""

umap_neighbors = [10, 20]
umap_components = [5, 10]
umap_min_dist = [0.0, 0.1, 0.5]

hdbscan_min_cluster = [50, 100, 150]
hdbscan_min_samples = [5, 10, 100]
hdbscan_cluter_eps = [0.1, 0.5]

embedding_models = ["all-MiniLM-L6-v2", "all-distilroberta-v1", "paraphrase-MiniLM-L6-v2"]

default_stopwords = []

results = []

In [7]:
from sentence_transformers import SentenceTransformer
embedding_all_Mini = SentenceTransformer("all-MiniLM-L6-v2", device='cuda')
embedding_roberta = SentenceTransformer("all-distilroberta-v1", device='cuda')
embedding_paraphrase = SentenceTransformer("paraphrase-MiniLM-L6-v2", device='cuda')

In [8]:
import nltk

try:
    # Tenta baixar o recurso 'punkt_tab'. Se já estiver baixado, ignora.
    nltk.download('punkt_tab')
except LookupError:
    # Se 'punkt_tab' não funcionar (depende da versão do NLTK), 
    # use o pacote 'punkt' mais genérico.
    # O seu traceback pede 'punkt_tab', então essa é a melhor aposta.
    # No entanto, se o NLTK for muito antigo, pode ser só 'punkt'.
    nltk.download('punkt')

# Se você estiver usando stopwords personalizadas, pode precisar disso também:
# nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/uselection/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
from itertools import product
from sklearn.feature_extraction.text import TfidfVectorizer
from meu_bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from nltk.tokenize import word_tokenize
import numpy as np
from sklearn.metrics import silhouette_score as compute_silhouette_score
from tqdm.auto import tqdm

for i, (n_neighbors, n_components, min_dist, min_cluster_size, min_samples, cluster_eps, embedding_model_name) in tqdm(
    enumerate(product(
        umap_neighbors,
        umap_components,
        umap_min_dist,
        hdbscan_min_cluster,
        hdbscan_min_samples,
        hdbscan_cluter_eps,
        embedding_models
    ), start=1),
    total=(len(umap_neighbors) * len(umap_components) * len(umap_min_dist) *
           len(hdbscan_min_cluster) * len(hdbscan_min_samples) * 
           len(hdbscan_cluter_eps) * len(embedding_models)),
    desc="Treinando modelos"
):    
    umap_params = {
        "n_neighbors": n_neighbors, 
        "n_components": n_components, 
        "min_dist": min_dist,
        "metric":'cosine',
        "random_state":42
    }

    hdbscan_params = {
        "min_cluster_size": min_cluster_size, 
        "min_samples": min_samples,
        "prediction_data":True,
        "cluster_selection_epsilon": cluster_eps
    }

    bertopic_params = {
        'language': 'english',
        #'verbose': True,
        'top_n_words': 20
    }

    if embedding_model_name == "all-MiniLM-L6-v2":
        embedding_model = embedding_all_Mini
    elif embedding_model_name == 'paraphrase-MiniLM-L6-v2':
        embedding_model = embedding_paraphrase
    else:
        embedding_model = embedding_roberta
    vectorizer = TfidfVectorizer(stop_words=default_stopwords,ngram_range=(1, 2))

    umap_model = UMAP(**umap_params)
    hdbscan_model = HDBSCAN(**hdbscan_params)

    model = BERTopic(
        **bertopic_params,
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True)
    )

    topics, probs = model.fit_transform(sample_docs)

    topic_words = model.get_topics() 
    # Extrai apenas as palavras para a Gensim
    topics_for_gensim = [[word for word, score in topic_words[topic_id]] 
                        for topic_id in topic_words if topic_id != -1]

    # 2. Tokenizar os documentos (necessário para Gensim)
    tokenized_docs = [word_tokenize(doc.lower()) for doc in sample_docs]

    # 3. Criar o Dicionário e Corpus BoW para Gensim
    dictionary = Dictionary(tokenized_docs)
    corpus_gensim = [dictionary.doc2bow(text) for text in tokenized_docs]

    # 4. Calcular Coerência
    coherence_model = CoherenceModel(
        topics=topics_for_gensim, 
        texts=tokenized_docs, 
        corpus=corpus_gensim,
        dictionary=dictionary, 
        coherence='c_v'  # Métrica C_V é a mais robusta
    )
    coherence_score = coherence_model.get_coherence()

        # --- Fluxo de cálculo da Diversidade (Proportion of Unique Words) ---
    topic_words = model.get_topics() 
    all_topic_words = []

    # Coleta todas as top N palavras, excluindo o tópico -1 (outliers)
    for topic_id in topic_words:
        if topic_id != -1:
            # Pega apenas as palavras (o primeiro elemento de cada tupla)
            words = [word for word, score in topic_words[topic_id]]
            all_topic_words.extend(words)
            
    total_words = len(all_topic_words)
    unique_words = len(set(all_topic_words))

    # Evita divisão por zero se o modelo não encontrar nenhum tópico (improvável no seu caso)
    if total_words > 0:
        diversity_score = unique_words / total_words
    else:
        diversity_score = 0

    try:
        # Tenta obter os embeddings reduzidos (o dado de clusterização)
        reduced_embeddings = model.hdbscan_model.data 
    except:
        # Se falhar, transforma manualmente:
        # 1. Gera embeddings de alta dimensão
        embeddings = embedding_model.encode(sample_docs) 
        # 2. Reduz a dimensão com o UMAP treinado
        reduced_embeddings = model.umap_model.transform(embeddings)

    # O topics (rótulos de cluster) é o resultado do model.fit_transform(sample_docs)
    # Filtra os outliers (-1) do HDBSCAN
    # O score da Silhueta geralmente é calculado *apenas* sobre os pontos que foram clusterizados
    # O sk-learn tem o parâmetro 'metric' que você já usou no UMAP ('cosine').

    # 1. Filtra os dados: remove o tópico -1 (outliers do HDBSCAN)
    mask = np.array(topics) != -1
    filtered_embeddings = reduced_embeddings[mask]
    filtered_topics = np.array(topics)[mask]

    # 2. Calcula o Silhouette Score (usa a métrica de distância 'cosine' que você usou no UMAP)
    if len(np.unique(filtered_topics)) > 1: # Precisa de pelo menos 2 clusters
        silhouette_score = compute_silhouette_score(
            filtered_embeddings, 
            filtered_topics, 
            metric='cosine' # Usa a métrica consistente com o UMAP
        )
    else:
        silhouette_score = 0.0 # Não é possível calcular Silhueta com 0 ou 1 cluster

    # 2. REGISTRO DOS RESULTADOS
    results.append({
        'n_neighbors': n_neighbors,
        'n_components': n_components,
        'min_dist': min_dist,
        'min_cluster_size': min_cluster_size,
        'min_samples': min_samples,
        'cluster_eps': cluster_eps,
        'embedding_model': embedding_model_name,
        'coherence': coherence_score,
        'diversity': diversity_score,
        'silhouette': silhouette_score
    })


Treinando modelos:   0%|          | 0/648 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [13]:
df_results = pd.DataFrame(results)
display(df_results)

,n_neighbors,n_components,min_dist,min_cluster_size,min_samples,cluster_eps,embedding_model,coherence,diversity,silhouette
0,10,5,0.0,50,5,0.1,all-MiniLM-L6-v2,0.496021,0.981250,0.673706
1,10,5,0.0,50,5,0.1,all-distilroberta-v1,0.483503,0.973214,0.598892
2,10,5,0.0,50,5,0.1,paraphrase-MiniLM-L6-v2,0.503240,0.997500,0.525819
3,10,5,0.0,50,5,0.5,all-MiniLM-L6-v2,0.493462,0.987500,-0.306492
4,10,5,0.0,50,5,0.5,all-distilroberta-v1,0.506426,0.987500,-0.358903
...,...,...,...,...,...,...,...,...,...,...
643,20,10,0.5,150,100,0.1,all-distilroberta-v1,0.624836,0.990000,0.611879
644,20,10,0.5,150,100,0.1,paraphrase-MiniLM-L6-v2,0.651854,0.970000,0.591654
645,20,10,0.5,150,100,0.5,all-MiniLM-L6-v2,0.657563,0.991667,0.743668
646,20,10,0.5,150,100,0.5,all-distilroberta-v1,0.624836,0.990000,0.611879


In [14]:
df_results['mean_score'] = df_results[['coherence', 'diversity', 'silhouette']].mean(axis=1)
display(df_results)

,n_neighbors,n_components,min_dist,min_cluster_size,min_samples,cluster_eps,embedding_model,coherence,diversity,silhouette,mean_score
0,10,5,0.0,50,5,0.1,all-MiniLM-L6-v2,0.496021,0.981250,0.673706,0.716992
1,10,5,0.0,50,5,0.1,all-distilroberta-v1,0.483503,0.973214,0.598892,0.685203
2,10,5,0.0,50,5,0.1,paraphrase-MiniLM-L6-v2,0.503240,0.997500,0.525819,0.675520
3,10,5,0.0,50,5,0.5,all-MiniLM-L6-v2,0.493462,0.987500,-0.306492,0.391490
4,10,5,0.0,50,5,0.5,all-distilroberta-v1,0.506426,0.987500,-0.358903,0.378341
...,...,...,...,...,...,...,...,...,...,...,...
643,20,10,0.5,150,100,0.1,all-distilroberta-v1,0.624836,0.990000,0.611879,0.742239
644,20,10,0.5,150,100,0.1,paraphrase-MiniLM-L6-v2,0.651854,0.970000,0.591654,0.737836
645,20,10,0.5,150,100,0.5,all-MiniLM-L6-v2,0.657563,0.991667,0.743668,0.797633
646,20,10,0.5,150,100,0.5,all-distilroberta-v1,0.624836,0.990000,0.611879,0.742239


In [15]:
df_results.to_csv("../data/bertopic_parameter_tuning_results_sample_3.csv", index=False)